### Comparaison des identifiants `customer_id` et `customer_unique_id`

In [4]:
from pathlib import Path

import pandas as pd

racine = Path.cwd()
if racine.name == "notebooks":
    racine = racine.parent

clients = pd.read_csv(
    racine / "data" / "raw" / "olist_customers_dataset.csv",
    usecols=["customer_id", "customer_unique_id"],
)

print("Nombre de lignes :", len(clients))
print("Valeurs distinctes par identifiant :")
print(clients.nunique())

print("Ecart constaté", clients.nunique()["customer_id"] - clients.nunique()["customer_unique_id"])

Nombre de lignes : 99441
Valeurs distinctes par identifiant :
customer_id           99441
customer_unique_id    96096
dtype: int64
Ecart constaté 3345


### Nombre de clients apparaissant plusieurs fois

In [2]:
lignes_par_client = clients.groupby("customer_unique_id").size()

print("Clients présents une seule fois :", (lignes_par_client == 1).sum())
print("Clients présents plusieurs fois :", (lignes_par_client > 1).sum())

Clients présents une seule fois : 93099
Clients présents plusieurs fois : 2997


### Rattachement de chaque commande à son client

In [5]:
from mktscore.data.loader import OlistDataLoader

commandes = OlistDataLoader(racine / "data" / "raw").load_orders()

commandes_clients = commandes.merge(
    clients,
    on="customer_id",
    how="left",
    validate="one_to_one",
)

print("Commandes après jointure :", len(commandes_clients))
print("Commandes sans client identifié :", commandes_clients["customer_unique_id"].isna().sum())

Commandes après jointure : 99441
Commandes sans client identifié : 0


Tout est nickel concernant la jointure.

### Examinons les statuts des commandes avant de compter les rachats

In [6]:
commandes_par_statut = commandes_clients["order_status"].value_counts(dropna=False)
print(commandes_par_statut)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


### Combien de commandes finalement livrées ont été passées chaque mois ?

In [8]:
commandes_livrees = commandes_clients.loc[
    commandes_clients["order_status"] == "delivered"
].copy()

mois_achat = commandes_livrees["order_purchase_timestamp"].dt.to_period("M")
commandes_par_mois = commandes_livrees.groupby(mois_achat).size().sort_index()

print(commandes_par_mois)
print("Total des commandes comptées :", commandes_par_mois.sum())

order_purchase_timestamp
2016-09       1
2016-10     265
2016-12       1
2017-01     750
2017-02    1653
2017-03    2546
2017-04    2303
2017-05    3546
2017-06    3135
2017-07    3872
2017-08    4193
2017-09    4150
2017-10    4478
2017-11    7289
2017-12    5513
2018-01    7069
2018-02    6555
2018-03    7003
2018-04    6798
2018-05    6749
2018-06    6099
2018-07    6159
2018-08    6351
Freq: M, dtype: int64
Total des commandes comptées : 96478


### Observons de plus près le pic du mois de Novembre

In [9]:
achats_novembre = commandes_livrees.loc[
    mois_achat == "2017-11", "order_purchase_timestamp"
]

commandes_par_jour = achats_novembre.dt.date.value_counts()
print(commandes_par_jour.nlargest(5))

order_purchase_timestamp
2017-11-24    1147
2017-11-25     487
2017-11-27     395
2017-11-26     382
2017-11-28     372
Name: count, dtype: int64


Le vendredi 24 novembre 2017 ressort avec 1 147 commandes passées ce jour-là puis livrées. Cela représente 15,7 % des 7 289 commandes du mois. Les cinq journées les plus actives sont consécutives, du 24 au 28 novembre. Ensemble, elles concentrent 38,2 % du volume mensuel. Le pic coïncide avec le Black Friday au Brésil. C’est une piste d’explication, mais ces comptages seuls ne permettent pas de mesurer l’effet des promotions.

### Analysons les délais de livraison

In [10]:
delais_livraison = (
    commandes_livrees["order_delivered_customer_date"]
    - commandes_livrees["order_purchase_timestamp"]
)
delais_jours = delais_livraison / pd.Timedelta(days=1)

print("Commandes sans délai calculable :", delais_jours.isna().sum())
print("Délais négatifs :", (delais_jours < 0).sum())
print("Délai médian en jours :", round(delais_jours.median(), 1))

Commandes sans délai calculable : 8
Délais négatifs : 0
Délai médian en jours : 10.2


Sur les 96 478 commandes au statut `delivered`, 96 470 ont un délai de livraison calculable. Aucun de ces délais n’est négatif. Le délai médian est d’environ 10,2 jours : la moitié de ces commandes ont été livrées en 10,2 jours ou moins après l’achat.

Les 8 autres commandes sont marquées `delivered`, mais leur date de livraison manque. Pandas les a automatiquement écartées du calcul de la médiane.

### Etude des rachats

In [11]:
nb_commandes_par_client = commandes_livrees.groupby("customer_unique_id")["order_id"].nunique()

print("Clients avec au moins une commande livrée :", len(nb_commandes_par_client))
print("Clients avec au moins deux commandes livrées :", (nb_commandes_par_client >= 2).sum())

Clients avec au moins une commande livrée : 93358
Clients avec au moins deux commandes livrées : 2801


Parmi les 93 358 clients ayant au moins une commande livrée, 2 801 en ont au moins deux, soit 3,00 %. Cette proportion porte sur toute la période observée. La question suivante est de savoir combien de temps s’est écoulé entre leur première et leur deuxième commande.

### Tri des commandes par client et par date d'achat puis calcul du délai entre les commandes

In [12]:
achats_tries = commandes_livrees.sort_values(
    ["customer_unique_id", "order_purchase_timestamp", "order_id"]
)

achats_par_client = achats_tries.groupby("customer_unique_id")
rang_achat = achats_par_client.cumcount() + 1
delai_depuis_achat_precedent = achats_par_client["order_purchase_timestamp"].diff()

delai_premier_rachat = delai_depuis_achat_precedent.loc[rang_achat == 2]
delai_median_jours = delai_premier_rachat.median() / pd.Timedelta(days=1)

print("Clients avec un délai calculé :", delai_premier_rachat.count())
print("Délai médian avant le deuxième achat, en jours :", round(delai_median_jours, 1))

Clients avec un délai calculé : 2801
Délai médian avant le deuxième achat, en jours : 29.0


Les 2 801 délais correspondent bien aux 2 801 clients ayant passé au moins deux commandes finalement livrées. Parmi ces clients, la moitié ont passé leur deuxième commande observée dans un délai d’environ 29 jours ou moins après la première.

### Combien sont revenus dans les 90 jours suivant leur premier achat ?

In [13]:
jour_zero = pd.Timedelta(0)
horizon = pd.Timedelta(days=90)

rachat_sous_90_jours = (
    (delai_premier_rachat > jour_zero)
    & (delai_premier_rachat <= horizon)
)

print("Deux premières commandes au même instant :", (delai_premier_rachat == jour_zero).sum())
print("Deuxième commande dans les 90 jours :", rachat_sous_90_jours.sum())
print("Deuxième commande après 90 jours :", (delai_premier_rachat > horizon).sum())

Deux premières commandes au même instant : 253
Deuxième commande dans les 90 jours : 1654
Deuxième commande après 90 jours : 894


Pour 1 654 des 2 801 clients, la deuxième commande a été passée après la première et dans les 90 jours suivants.

### Analyse plus fine, tenant compte des client ayant effectué leurs deux premières commandes le même jour et qui sont revenus après, sous 90 jours

In [14]:
instants_achat = achats_tries.drop_duplicates(
    subset=["customer_unique_id", "order_purchase_timestamp"]
)

instants_par_client = instants_achat.groupby("customer_unique_id")
rang_instant = instants_par_client.cumcount() + 1
delais = instants_par_client["order_purchase_timestamp"].diff()

delai_premier_retour = delais.loc[rang_instant == 2]

print("Clients revenus à un instant ultérieur :", delai_premier_retour.count())
print("Clients revenus dans les 90 jours :", (delai_premier_retour <= horizon).sum())

Clients revenus à un instant ultérieur : 2565
Clients revenus dans les 90 jours : 1666


1 666 clients ont passé une nouvelle commande à un instant distinct dans les 90 jours suivant leur premier achat observé, en comptant uniquement les commandes finalement livrées.

### Taux de rachat à 90 jours pour une cohorte précise

In [16]:
premier_achat = instants_par_client["order_purchase_timestamp"].min()
clients_2017 = premier_achat.loc[premier_achat.dt.year == 2017].index

# Associer le délai de retour à l'identifiant du client.
retours = instants_achat.assign(delai=delais).loc[rang_instant == 2]
delai_par_client = retours.set_index("customer_unique_id")["delai"]

delais_2017 = delai_par_client.reindex(clients_2017)
nb_eligibles = len(clients_2017)
nb_rachats_90j = (delais_2017 <= horizon).sum()

print("Clients de la cohorte 2017 :", nb_eligibles)
print("Clients revenus à un instant ultérieur :", delais_2017.count())
print("Clients revenus dans les 90 jours :", nb_rachats_90j)
print(f"Taux de rachat à 90 jours : {nb_rachats_90j / nb_eligibles:.2%}")

Clients de la cohorte 2017 : 42131
Clients revenus à un instant ultérieur : 1629
Clients revenus dans les 90 jours : 896
Taux de rachat à 90 jours : 2.13%


Parmi les 42 131 clients dont le premier achat observé parmi les commandes finalement livrées date de 2017, 896 ont repassé commande dans les 90 jours, soit 2,13 %. Le délai est mesuré entre les dates d’achat. Deux commandes enregistrées au même instant comptent comme une seule occasion d’achat.

### Score CRM

#### Récence

In [17]:
date_scoring = pd.Timestamp("2018-01-01 00:00:00")

historique = commandes_livrees.loc[
    (commandes_livrees["order_purchase_timestamp"] <= date_scoring)
    & (commandes_livrees["order_delivered_customer_date"] <= date_scoring)
]

dernier_achat = historique.groupby("customer_unique_id")["order_purchase_timestamp"].max()
clients_scoring = dernier_achat.rename("dernier_achat").to_frame()

clients_scoring["recence_jours"] = (
    date_scoring - clients_scoring["dernier_achat"]
).dt.days

print("Clients éligibles :", len(clients_scoring))
print(clients_scoring.head())

Clients éligibles : 39964
                                       dernier_achat  recence_jours
customer_unique_id                                                 
0000f46a3911fa3c0805444483337064 2017-03-10 21:05:03            296
0000f6ccb0745a6a4b88665a16c9f078 2017-10-12 20:29:41             80
0004aac84e0df4da2b147fca70cf8255 2017-11-14 19:45:42             47
0005e1862207bf6ccc02e4228effd9a0 2017-03-04 23:32:12            302
0006fdc98a402fceb4eb0ee528f6a8d4 2017-07-18 09:23:10            166


#### Ajout de la cible `rachat_90j`

In [18]:
fin_fenetre = date_scoring + pd.Timedelta(days=90)

achats_dans_la_fenetre = (
    (commandes_livrees["order_purchase_timestamp"] > date_scoring)
    & (commandes_livrees["order_purchase_timestamp"] <= fin_fenetre)
)

clients_avec_rachat = commandes_livrees.loc[
    achats_dans_la_fenetre, "customer_unique_id"
].unique()

clients_scoring["rachat_90j"] = (
    clients_scoring.index.isin(clients_avec_rachat).astype(int)
)

print(clients_scoring["rachat_90j"].value_counts().sort_index())
print(f"Taux de rachat à 90 jours : {clients_scoring['rachat_90j'].mean():.2%}")

rachat_90j
0    39674
1      290
Name: count, dtype: int64
Taux de rachat à 90 jours : 0.73%


#### Groupes de clients selon la récence

In [19]:
tranches_recence = pd.cut(
    clients_scoring["recence_jours"],
    bins=[0, 31, 91, 181, float("inf")],
    labels=["0 à 30 jours", "31 à 90 jours", "91 à 180 jours", "181 jours et plus"],
    right=False,
)

bilan_recence = (
    clients_scoring.groupby(tranches_recence, observed=True)["rachat_90j"]
    .agg(clients="size", clients_avec_rachat="sum")
)

bilan_recence["taux_rachat_pct"] = (
    100 * bilan_recence["clients_avec_rachat"] / bilan_recence["clients"]
).round(2)

print(bilan_recence)

                   clients  clients_avec_rachat  taux_rachat_pct
recence_jours                                                   
0 à 30 jours          3292                   34             1.03
31 à 90 jours        11092                   79             0.71
91 à 180 jours       11657                   94             0.81
181 jours et plus    13923                   83             0.60


Au 1er janvier 2018, les clients dont le dernier achat remontait à 30 jours ou moins avaient le taux de rachat observé le plus élevé sur les 90 jours suivants (1,03 %, contre 0,73 % pour l’ensemble des clients éligibles). Ce groupe représentait toutefois seulement 34 des 290 clients revenus. La récence seule aurait donc laissé de côté la majorité des futurs acheteurs.

### Conclusion

Au 1er janvier 2018, les clients dont le dernier achat remontait à 30 jours ou moins présentaient le taux de rachat à 90 jours le plus élevé (1,03 %). Ce résultat peut guider le choix d’une audience à tester. Pour estimer combien de rachats un message CRM génère en plus des rachats spontanés, il faudrait comparer cette audience à un groupe témoin constitué par tirage au sort avant l’envoi.